# Mini project 3 — First Part: Default simulation and collateral cash flows

This notebook simulates **1,000 cases for 10 bonds**, converts default times to quarterly cash flows, and sums the collateral available each quarter. Cases are sorted from worst to best by the number of defaults within five years: case 1 has the most defaults, and case 1,000 has the fewest. Change `SCENARIO_ID` to inspect any ranked case.

**Conventions:** Dollar amounts are in USD; times are in years. Each bond pays a $150,000 quarterly coupon. On default, it pays recovery of 40% of face value ($4 million) once, at the end of the quarter containing default. It pays no coupon in that quarter and nothing afterward. A default exactly on a payment date takes precedence over that payment. There is no accrued coupon. The document specifies LGD but does not specify precise recovery timing, so these are explicit modeling assumptions.

**Correlation requirement:** Every pair of uncensored default times has a target population Pearson correlation of **0.20**. We calibrate the correlation between normal scores, construct their equicorrelation matrix, and use its Cholesky factor to generate correlated normal draws. This latent correlation is calibrated to meet the target after the transformation to default times. The 1,000-case sample correlations will fluctuate around the population target; sorting rows by default count does not change those correlations.

In [36]:
import numpy as np
import pandas as pd
from scipy.special import ndtr, log_ndtr, roots_hermitenorm
from scipy.optimize import brentq
from IPython.display import display

N_SCENARIOS = 1_000
N_BONDS = 10
MATURITY_YEARS = 5
PAYMENTS_PER_YEAR = 4
FACE_VALUE = 10_000_000.0
ANNUAL_COUPON_RATE = 0.06
ANNUAL_DEFAULT_PROBABILITY = 0.04
LGD = 0.60
TARGET_DEFAULT_TIME_CORRELATION = 0.20
SEED = 6103

DT = 1 / PAYMENTS_PER_YEAR # time between payments in years
N_QUARTERS = MATURITY_YEARS * PAYMENTS_PER_YEAR
payment_times = np.arange(1, N_QUARTERS + 1) * DT
quarter_labels = [f"Q{q}" for q in range(1, N_QUARTERS + 1)]
bond_labels = [f"Bond {b}" for b in range(1, N_BONDS + 1)]
scenario_ids = pd.Index(range(1, N_SCENARIOS + 1), name="Case")
QUARTERLY_COUPON = FACE_VALUE * ANNUAL_COUPON_RATE * DT
RECOVERY = FACE_VALUE * (1 - LGD)

display(pd.Series({
    "Scenarios": N_SCENARIOS, "Bonds": N_BONDS,
    "Quarters": N_QUARTERS, "Face value per bond ($)": FACE_VALUE,
    "Quarterly coupon per bond ($)": QUARTERLY_COUPON,
    "Recovery per default ($)": RECOVERY,
}, name="Model inputs").to_frame())

,Model inputs
Scenarios,1000.0
Bonds,10.0
Quarters,20.0
Face value per bond ($),10000000.0
Quarterly coupon per bond ($),150000.0
Recovery per default ($),4000000.0


## 1. Convert annual default probability to a hazard rate

A **hazard rate** measures how quickly default can occur, **given that the bond has not already defaulted**. It describes default risk continuously through time.

It differs slightly from an annual default probability:

- **Annual default probability of 4%:** a surviving bond has a 4% chance of defaulting over the next full year.
- **Hazard rate:** the instantaneous rate of default among bonds that have survived so far. It is a rate per year, not itself a one-year probability.

With a constant hazard rate $\lambda$, survival through time $t$ is $S(t)=e^{-\lambda t}$, so

$$P(\text{default within }t\text{ years})=1-e^{-\lambda t}.$$

To make the one-year default probability equal to 4%, we solve:

$$0.04=1-e^{-\lambda}
\quad\Longrightarrow\quad
\lambda=-\ln(0.96)=0.0408219945\text{ per year}.$$

The notebook therefore uses a **hazard rate of about 4.0822% per year**, distinct from the 4% one-year probability. This lets us generate default times anywhere along the timeline—for example, at 0.6 years or 3.2 years—while preserving the required 4% annual default probability.

The conditional quarterly default probability is $1-e^{-\lambda/4}$, and the five-year cumulative default probability is $1-e^{-5\lambda}$.

In [37]:
hazard_rate = -np.log1p(-ANNUAL_DEFAULT_PROBABILITY)
quarterly_default_probability = -np.expm1(-hazard_rate * DT)
five_year_default_probability = -np.expm1(-hazard_rate * MATURITY_YEARS)
assert np.isclose(-np.expm1(-hazard_rate), ANNUAL_DEFAULT_PROBABILITY)
print(f"Annual hazard rate: {hazard_rate:.10f} ({hazard_rate:.6%})")
print(f"Conditional quarterly default probability: {quarterly_default_probability:.6%}")
print(f"Five-year cumulative default probability: {five_year_default_probability:.6%}")

Annual hazard rate: 0.0408219945 (4.082199%)
Conditional quarterly default probability: 1.015360%
Five-year cumulative default probability: 18.462730%


## 2. Calibrate correlation and generate default times

The requirement applies to **default times**, so we first find the normal-score correlation $\rho$ that produces a population default-time correlation of 0.20. Build the $10\times10$ equicorrelation matrix $R$ with ones on its diagonal and calibrated $\rho$ in every off-diagonal entry, then factor it as $R=LL^T$.

For each bond, generate 1,000 independent standard-normal draws. **Moment-match each column** by subtracting its sample mean and dividing by its sample standard deviation (using `ddof=1`). This gives each independent-normal column sample mean 0 and sample standard deviation 1; its cross-column sample correlations remain close to, but not necessarily exactly, zero.

For scenario rows, apply the Cholesky factor as $X=ZL^T$. Then transform the correlated normals through the standard normal CDF and the inverse exponential CDF:

$$U_{s,i}=\Phi(X_{s,i}),\qquad
\tau_{s,i}=F^{-1}(U_{s,i})=-\frac{\ln(1-U_{s,i})}{\lambda}.$$

The generated times are sorted from worst to best by the number of defaults within five years. The same seed and settings reproduce the same draws, moment-matched normals, default times, and case ordering.

**How correlation calibration works.** Calibration is separate from simulating the 10 bonds. For each candidate $\rho$, we numerically integrate the distribution of just one correlated pair of normal scores to calculate the implied population correlation of their exponential default times. In that two-dimensional integral, $A$ and $B$ are independent standard normals and

$$X_i=A,\qquad X_j=\rho A+\sqrt{1-\rho^2}B.$$

This is simply a convenient way to evaluate a bivariate normal expectation (equivalently, Cholesky for a $2\times2$ correlation matrix); it does **not** generate scenarios or replace the 10-dimensional Cholesky simulation below.

By symmetry of the standard normal, $-\ln(1-\Phi(X))=-\ln\Phi(-X)$ has the same joint distribution as $g(X)=-\ln\Phi(X)$ when the normal-score correlation is $\rho$. For a standard normal $X$, $g(X)$ is exponential with mean 1 and variance 1. Therefore,

$$\operatorname{Corr}(\tau_i,\tau_j)
=\operatorname{Corr}(g(X_i),g(X_j))
=E[g(X_i)g(X_j)]-1.$$

Dividing by the positive hazard rate does not change correlation. Gauss–Hermite quadrature numerically evaluates this expectation, and a root solver finds the $\rho$ for which the population default-time correlation is 0.20. We check the answer with a higher quadrature order. Moment matching improves the finite simulation's marginal normal moments, but does not force the finite-sample default-time correlations to equal 0.20 exactly; those are reported as diagnostics.

The target applies to full default times, including those beyond five years, not to default indicators or times capped at maturity. The fixed seed preserves reproducibility of the random draws.

In [38]:
def implied_default_time_correlation(rho, quadrature_order=48):
    """Population correlation of exponential times from correlated normals."""
    if not 0 <= rho <= 1:
        raise ValueError("The normal-score correlation must lie between 0 and 1.")
    nodes, weights = roots_hermitenorm(quadrature_order)
    weights = weights / np.sqrt(2 * np.pi)  # Standard normal expectation weights.
    x = nodes[:, None]
    y = rho * x + np.sqrt(1 - rho**2) * nodes[None, :]
    # log_ndtr evaluates log(Phi(x)) stably even far into the normal tail.
    g_x, g_y = -log_ndtr(x), -log_ndtr(y)
    expected_product = np.sum(weights[:, None] * weights[None, :] * g_x * g_y)
    return float(expected_product - 1.0)

if not 0 <= TARGET_DEFAULT_TIME_CORRELATION <= 1:
    raise ValueError("This Gaussian-copula model supports targets from 0 to 1.")
if TARGET_DEFAULT_TIME_CORRELATION in (0, 1):
    LATENT_RHO = float(TARGET_DEFAULT_TIME_CORRELATION)
else:
    LATENT_RHO = brentq(
        lambda rho: implied_default_time_correlation(rho) - TARGET_DEFAULT_TIME_CORRELATION,
        0.0, 1.0, xtol=1e-12,
    )

# Independently increase integration accuracy to check convergence.
verified_population_correlation = implied_default_time_correlation(LATENT_RHO, quadrature_order=80)
np.testing.assert_allclose(verified_population_correlation, TARGET_DEFAULT_TIME_CORRELATION,
                           rtol=0, atol=1e-9)
np.testing.assert_allclose(implied_default_time_correlation(0.0, 80), 0.0, rtol=0, atol=1e-9)
np.testing.assert_allclose(implied_default_time_correlation(1.0, 80), 1.0, rtol=0, atol=1e-9)
print(f"Target default-time correlation: {TARGET_DEFAULT_TIME_CORRELATION:.6f}")
print(f"Calibrated normal-score correlation: {LATENT_RHO:.10f}")
print(f"Verified population default-time correlation: {verified_population_correlation:.10f}")
print(f"For comparison, normal-score correlation 0.20 implies default-time correlation "
      f"{implied_default_time_correlation(0.20, 80):.6f}.")

Target default-time correlation: 0.200000
Calibrated normal-score correlation: 0.2332349314
Verified population default-time correlation: 0.2000000000
For comparison, normal-score correlation 0.20 implies default-time correlation 0.170302.


In [39]:
rng = np.random.default_rng(SEED)
normal_correlation = np.full((N_BONDS, N_BONDS), LATENT_RHO)
np.fill_diagonal(normal_correlation, 1.0)
cholesky_factor = np.linalg.cholesky(normal_correlation)
independent_normals = rng.standard_normal((N_SCENARIOS, N_BONDS))
independent_normals = (
    independent_normals - independent_normals.mean(axis=0)
) / independent_normals.std(axis=0, ddof=1)
latent_scores = independent_normals @ cholesky_factor.T
sample_latent_correlation = np.corrcoef(latent_scores, rowvar=False)
uniforms = ndtr(latent_scores)
# Guard against numerical rounding to exactly 0 or 1.
uniforms = np.clip(uniforms, np.finfo(float).tiny, np.nextafter(1.0, 0.0))
default_times = -np.log1p(-uniforms) / hazard_rate

# Rank cases by the number of defaults within maturity; stable ties retain seeded draw order.
default_counts = (default_times <= MATURITY_YEARS).sum(axis=1)
scenario_order = np.argsort(-default_counts, kind="stable")
independent_normals = independent_normals[scenario_order]
latent_scores = latent_scores[scenario_order]
uniforms = uniforms[scenario_order]
default_times = default_times[scenario_order]
default_counts = default_counts[scenario_order]

default_times_df = pd.DataFrame(default_times, index=scenario_ids, columns=bond_labels)

assert default_times.shape == (N_SCENARIOS, N_BONDS)
assert np.all(np.isfinite(default_times)) and np.all(default_times > 0)
assert np.allclose(independent_normals.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(independent_normals.std(axis=0, ddof=1), 1.0, atol=1e-12)
assert np.all(np.diff(default_counts) <= 0)
assert default_counts[0] == default_counts.max()
assert default_counts[-1] == default_counts.min()
print("Default times in years, first five cases ranked worst to best:")
display(default_times_df.head().round(4))
print(f"Defaults in case 1 (worst): {default_counts[0]} / {N_BONDS}")
print(f"Defaults in case {N_SCENARIOS} (best): {default_counts[-1]} / {N_BONDS}")
print(f"Defaults within five years: {default_counts.sum():,} / {default_times.size:,}")
print(f"Average sample latent-normal correlation: "
      f"{sample_latent_correlation[np.triu_indices(N_BONDS, k=1)].mean():.4f} "
      f"(target {LATENT_RHO:.4f})")

Default times in years, first five cases ranked worst to best:


,Bond 1,Bond 2,Bond 3,Bond 4,Bond 5,Bond 6,Bond 7,Bond 8,Bond 9,Bond 10
Case,,,,,,,,,,
1,0.1755,0.3260,2.0744,1.5734,3.7078,0.4596,0.0488,4.3922,18.2728,0.7611
2,4.6125,1.4418,1.3127,3.1634,1.4174,0.7072,0.2324,4.0991,14.5942,0.9683
3,0.1603,1.4381,0.1528,0.6211,1.4186,6.8929,1.8563,0.6154,0.2922,3.1394
4,6.4830,2.6528,0.9545,33.4137,1.5111,0.0283,4.4992,3.4479,3.5883,0.4207
5,1.6990,9.9063,2.3841,4.8846,1.4421,0.8256,0.8919,0.9452,9.8587,0.2782


Defaults in case 1 (worst): 9 / 10
Defaults in case 1000 (best): 0 / 10
Defaults within five years: 1,857 / 10,000
Average sample latent-normal correlation: 0.2470 (target 0.2332)


## 3. Convert default times to quarterly bond cash flows

For payment date $t_q=q/4$, coupon $C=150{,}000$, face value $F=10{,}000{,}000$, and recovery $R=4{,}000{,}000$,

$$CF_{s,i,q}=
C\,\mathbf{1}_{\{\tau_{s,i}>t_q\}}
+R\,\mathbf{1}_{\{t_{q-1}<\tau_{s,i}\leq t_q\}}
+F\,\mathbf{1}_{\{q=20,\ \tau_{s,i}>5\}}.$$

The first term pays coupons while the bond survives; the second pays recovery only in the default quarter; the third repays principal at maturity only for surviving bonds. Recovery is allocated to a quarterly bucket without discounting. The project’s 9% YTM and 1% risk-free rate are inputs to later valuation, not to these undiscounted contractual cash flows.

In [40]:
def bond_cash_flow_components(times):
    """Map positive default times shaped (scenario, bond) to USD arrays.

    Each returned array has shape (scenario, bond, quarter).
    Positive infinity is allowed to represent no default in a test case.
    """
    times = np.asarray(times, dtype=float)
    if times.ndim != 2 or np.isnan(times).any() or np.any(times <= 0):
        raise ValueError("Default times must be a positive 2-D array; +inf is allowed.")
    tau = times[:, :, None]
    alive = tau > payment_times
    defaults_this_quarter = (tau > (payment_times - DT)) & (tau <= payment_times)
    coupons = QUARTERLY_COUPON * alive
    recoveries = RECOVERY * defaults_this_quarter
    principal = np.zeros_like(coupons)
    principal[:, :, -1] = FACE_VALUE * (times > MATURITY_YEARS)
    return coupons, recoveries, principal

coupon_cash_flows, recovery_cash_flows, principal_cash_flows = bond_cash_flow_components(default_times)
bond_cash_flows = coupon_cash_flows + recovery_cash_flows + principal_cash_flows
assert bond_cash_flows.shape == (N_SCENARIOS, N_BONDS, N_QUARTERS)
assert np.all(bond_cash_flows >= 0)
np.testing.assert_allclose(
    recovery_cash_flows.sum(axis=2), RECOVERY * (default_times <= MATURITY_YEARS)
)
assert np.all(bond_cash_flows[default_times[:, :, None] <= (payment_times - DT)] == 0)
print("Bond cash flow array shape:", bond_cash_flows.shape)

Bond cash flow array shape: (1000, 10, 20)


## 4. Sum the 10 bonds and inspect a case

`collateral_cash_flows[s, q]` is the sum across all bonds for one case and quarter. Case labels are ranked 1–1,000 by defaults within five years, descending; case 1 is worst. NumPy indices are 0–999. Set `SCENARIO_ID` in the inspection cell to choose any ranked case. The inspection table shows bond cash flows and separate collateral coupon, recovery, and principal totals. All table values below are in **millions of dollars**.

In [41]:
collateral_cash_flows = bond_cash_flows.sum(axis=1)
collateral_cash_flows_df = pd.DataFrame(
    collateral_cash_flows, index=scenario_ids, columns=quarter_labels
)
assert collateral_cash_flows.shape == (N_SCENARIOS, N_QUARTERS)

def inspect_scenario(scenario_id):
    if not isinstance(scenario_id, (int, np.integer)) or not 1 <= scenario_id <= N_SCENARIOS:
        raise ValueError(f"Choose an integer case ID from 1 to {N_SCENARIOS}.")
    s = scenario_id - 1
    default_quarters = [
        int(np.searchsorted(payment_times, t, side="left") + 1)
        if t <= MATURITY_YEARS else None for t in default_times[s]
    ]
    details = pd.DataFrame({
        "Default time (years)": default_times[s],
        "Default quarter": pd.array(default_quarters, dtype="Int64"),
        "Survives maturity": default_times[s] > MATURITY_YEARS,
    }, index=bond_labels)
    cash = pd.DataFrame(bond_cash_flows[s].T / 1e6, index=quarter_labels, columns=bond_labels)
    cash["Coupons"] = coupon_cash_flows[s].sum(axis=0) / 1e6
    cash["Recoveries"] = recovery_cash_flows[s].sum(axis=0) / 1e6
    cash["Principal"] = principal_cash_flows[s].sum(axis=0) / 1e6
    cash["Collateral total"] = collateral_cash_flows[s] / 1e6
    print(f"Case {scenario_id} (ranked worst to best): {default_counts[s]} defaults within five years; cash flows in $ millions.")
    display(details.round(4))
    display(cash.round(4))
    return details, cash

SCENARIO_ID = 1  # Select any ranked case from 1 (most defaults) to 1000 (fewest defaults).
scenario_details, scenario_cash_flows = inspect_scenario(SCENARIO_ID)

Case 1 (ranked worst to best): 9 defaults within five years; cash flows in $ millions.


,Default time (years),Default quarter,Survives maturity
Bond 1,0.1755,1,False
Bond 2,0.3260,2,False
Bond 3,2.0744,9,False
Bond 4,1.5734,7,False
Bond 5,3.7078,15,False
Bond 6,0.4596,2,False
Bond 7,0.0488,1,False
Bond 8,4.3922,18,False
Bond 9,18.2728,<NA>,True
Bond 10,0.7611,4,False


,Bond 1,Bond 2,Bond 3,Bond 4,Bond 5,Bond 6,Bond 7,Bond 8,Bond 9,Bond 10,Coupons,Recoveries,Principal,Collateral total
Q1,4.0,0.15,0.15,0.15,0.15,0.15,4.0,0.15,0.15,0.15,1.20,8.0,0.0,9.20
Q2,0.0,4.00,0.15,0.15,0.15,4.00,0.0,0.15,0.15,0.15,0.90,8.0,0.0,8.90
Q3,0.0,0.00,0.15,0.15,0.15,0.00,0.0,0.15,0.15,0.15,0.90,0.0,0.0,0.90
Q4,0.0,0.00,0.15,0.15,0.15,0.00,0.0,0.15,0.15,4.00,0.75,4.0,0.0,4.75
Q5,0.0,0.00,0.15,0.15,0.15,0.00,0.0,0.15,0.15,0.00,0.75,0.0,0.0,0.75
Q6,0.0,0.00,0.15,0.15,0.15,0.00,0.0,0.15,0.15,0.00,0.75,0.0,0.0,0.75
Q7,0.0,0.00,0.15,4.00,0.15,0.00,0.0,0.15,0.15,0.00,0.60,4.0,0.0,4.60
Q8,0.0,0.00,0.15,0.00,0.15,0.00,0.0,0.15,0.15,0.00,0.60,0.0,0.0,0.60
Q9,0.0,0.00,4.00,0.00,0.15,0.00,0.0,0.15,0.15,0.00,0.45,4.0,0.0,4.45
Q10,0.0,0.00,0.00,0.00,0.15,0.00,0.0,0.15,0.15,0.00,0.45,0.0,0.0,0.45


## 5. Validate cash flows by hand

The following deliberately constructed scenarios give transparent checks independent of the random draws. Amounts are in $ millions.

| Scenario | Hand calculation | Total over 20 quarters |
|---|---|---:|
| No defaults | Q1–Q19: 10 × 0.15 = 1.50 each; Q20: 10 × 10.15 = 101.50 | 130.00 |
| Bond 1 defaults at 0.10 years | Q1: 9 × 0.15 + 4 = 5.35; Q2–Q19: 1.35 each; Q20: 9 × 10.15 = 91.35 | 121.00 |
| Bond 1 defaults at 0.60 years | Q1–Q2: 1.50 each; Q3: 1.35 + 4 = 5.35; Q4–Q19: 1.35 each; Q20: 91.35 | 121.30 |
| All bonds default at 4.90 years | Q1–Q19: 1.50 each; Q20: 10 × 4 = 40.00 | 68.50 |

Boundary checks also verify that default at exactly 0.25 years replaces the Q1 coupon with recovery, and default at exactly five years replaces the final coupon and principal with recovery.

In [42]:
manual_times = np.full((4, N_BONDS), np.inf)
manual_times[1, 0] = 0.10
manual_times[2, 0] = 0.60
manual_times[3, :] = 4.90
manual_actual = sum(bond_cash_flow_components(manual_times)).sum(axis=1)

# Explicit expected schedules from the hand calculations above, in dollars.
manual_expected = np.array([
    [1.50] * 19 + [101.50],
    [5.35] + [1.35] * 18 + [91.35],
    [1.50, 1.50, 5.35] + [1.35] * 16 + [91.35],
    [1.50] * 19 + [40.00],
]) * 1e6
np.testing.assert_allclose(manual_actual, manual_expected, rtol=0, atol=1e-6)
boundary_actual = sum(bond_cash_flow_components(np.array([[0.25, 5.0, 5.000001]])))
boundary_expected = np.array([
    [4.00] + [0.00] * 19,
    [0.15] * 19 + [4.00],
    [0.15] * 19 + [10.15],
]) * 1e6
np.testing.assert_allclose(boundary_actual[0], boundary_expected, rtol=0, atol=1e-6)
display(pd.DataFrame({
    "Expected total ($m)": manual_expected.sum(axis=1) / 1e6,
    "Simulated total ($m)": manual_actual.sum(axis=1) / 1e6,
    "All 20 quarters match": np.isclose(manual_actual, manual_expected).all(axis=1),
}, index=["No defaults", "One default in Q1", "One default in Q3", "All default in Q20"]))
print("All manual schedules and payment-date boundary checks passed.")

,Expected total ($m),Simulated total ($m),All 20 quarters match
No defaults,130.0,130.0,True
One default in Q1,121.0,121.0,True
One default in Q3,121.3,121.3,True
All default in Q20,68.5,68.5,True


All manual schedules and payment-date boundary checks passed.


### Cross-check three actual simulated scenarios

For three scenarios, the next cell independently walks through each bond and each payment date. It also counts surviving bonds and new defaults each quarter to verify the arithmetic: coupon cash = survivor count × $150,000; recovery cash = new defaults × $4 million; final principal = maturity survivors × $10 million. This catches incorrect bond aggregation and recovery timing.

In [43]:
def cash_flows_by_loop(times):
    result = np.zeros((len(times), N_QUARTERS))
    for b, tau in enumerate(times):
        for q, t in enumerate(payment_times):
            if tau <= t:
                result[b, q] = RECOVERY
                break
            result[b, q] = QUARTERLY_COUPON
            if q == N_QUARTERS - 1:
                result[b, q] += FACE_VALUE
    return result

default_counts = (default_times <= MATURITY_YEARS).sum(axis=1)
selected_indices = np.unique([0, int(np.argmin(default_counts)), int(np.argmax(default_counts))]).tolist()
for candidate in range(N_SCENARIOS):
    if len(selected_indices) >= 3:
        break
    if candidate not in selected_indices:
        selected_indices.append(candidate)

audit_rows = []
for s in selected_indices:
    reference = cash_flows_by_loop(default_times[s])
    np.testing.assert_allclose(bond_cash_flows[s], reference, rtol=0, atol=1e-6)
    np.testing.assert_allclose(collateral_cash_flows[s], reference.sum(axis=0), rtol=0, atol=1e-6)
    for q, t in enumerate(payment_times):
        surviving = int(np.sum(default_times[s] > t))
        new_defaults = int(np.sum((default_times[s] > t - DT) & (default_times[s] <= t)))
        hand_total = surviving * QUARTERLY_COUPON + new_defaults * RECOVERY
        if q == N_QUARTERS - 1:
            hand_total += surviving * FACE_VALUE
        np.testing.assert_allclose(hand_total, collateral_cash_flows[s, q], rtol=0, atol=1e-6)
        if q in (0, N_QUARTERS - 1) or new_defaults:
            audit_rows.append([s + 1, q + 1, surviving, new_defaults, hand_total / 1e6, collateral_cash_flows[s, q] / 1e6])

display(pd.DataFrame(audit_rows, columns=[
    "Case", "Quarter", "Survivors", "New defaults", "Hand total ($m)", "Model total ($m)"
]))
print(f"All bond and collateral cash flows checked for cases {[s + 1 for s in selected_indices]}.")

,Case,Quarter,Survivors,New defaults,Hand total ($m),Model total ($m)
0,1,1,8,2,9.20,9.20
1,1,2,6,2,8.90,8.90
2,1,4,5,1,4.75,4.75
3,1,7,4,1,4.60,4.60
4,1,9,3,1,4.45,4.45
5,1,15,2,1,4.30,4.30
6,1,18,1,1,4.15,4.15
7,1,20,1,0,10.15,10.15
8,723,1,10,0,1.50,1.50
9,723,20,10,0,101.50,101.50


All bond and collateral cash flows checked for cases [1, 723, 2].


## 6. Simulation diagnostics and handoff

As a further check, expected collateral cash flow at quarter $q$ is

$$E[CF_q]=10\left[C S(t_q)+R\{S(t_{q-1})-S(t_q)\}
+F\,\mathbf{1}_{\{q=20\}}S(5)\right].$$

Correlation affects dispersion but not this expectation. The sample means will differ from the theoretical values because there are only 1,000 scenarios. Standard errors are computed across independent **scenario totals**, preserving dependence between bonds within a scenario. Correlation diagnostics are descriptive, not requirements that a finite sample equal its population target exactly.

In [44]:
survival = np.exp(-hazard_rate * payment_times)
previous_survival = np.exp(-hazard_rate * (payment_times - DT))
expected_collateral = N_BONDS * (
    QUARTERLY_COUPON * survival + RECOVERY * (previous_survival - survival)
)
expected_collateral[-1] += N_BONDS * FACE_VALUE * survival[-1]
simulation_mean = collateral_cash_flows.mean(axis=0)
simulation_se = collateral_cash_flows.std(axis=0, ddof=1) / np.sqrt(N_SCENARIOS)
diagnostics = pd.DataFrame({
    "Theoretical mean ($m)": expected_collateral / 1e6,
    "Simulation mean ($m)": simulation_mean / 1e6,
    "Monte Carlo SE ($m)": simulation_se / 1e6,
    "Difference / SE": np.divide(simulation_mean - expected_collateral, simulation_se,
                                 out=np.full(N_QUARTERS, np.nan), where=simulation_se > 0),
}, index=quarter_labels)
display(diagnostics.round(4))

pairs = np.triu_indices(N_BONDS, k=1)
mean_latent_correlation = np.corrcoef(latent_scores, rowvar=False)[pairs].mean()
default_time_correlation_matrix = np.corrcoef(default_times, rowvar=False)
pairwise_time_correlations = default_time_correlation_matrix[pairs]
mean_time_correlation = pairwise_time_correlations.mean()
print(f"Average sample latent-score correlation: {mean_latent_correlation:.4f} (calibrated target {LATENT_RHO:.4f})")
print(f"Population default-time correlation: {verified_population_correlation:.6f} "
      f"(target {TARGET_DEFAULT_TIME_CORRELATION:.2f})")
print(f"Average sample default-time correlation: {mean_time_correlation:.4f}")
print(f"Range across the 45 sample bond-pair correlations: "
      f"{pairwise_time_correlations.min():.4f} to {pairwise_time_correlations.max():.4f}")
print("Sample correlations vary because there are 1,000 scenarios; calibration targets the population.")
display(pd.DataFrame(default_time_correlation_matrix, index=bond_labels, columns=bond_labels).round(3))
print(f"Five-year default frequency: {default_counts.mean() / N_BONDS:.4%}")
print(f"Theoretical five-year default probability: {five_year_default_probability:.4%}")
print(f"Mean number of defaults per scenario: {default_counts.mean():.3f}")
print(f"Mean total collateral cash ($m): {collateral_cash_flows.sum(axis=1).mean() / 1e6:.4f}")
print("Validation complete: all deterministic cash-flow assertions passed.")

,Theoretical mean ($m),Simulation mean ($m),Monte Carlo SE ($m),Difference / SE
Q1,1.8909,1.8966,0.0433,0.1302
Q2,1.8717,1.8888,0.0418,0.4089
Q3,1.8527,1.8615,0.0380,0.2313
Q4,1.8339,1.9194,0.0432,1.9763
Q5,1.8153,1.8358,0.0405,0.5060
Q6,1.7968,1.8202,0.0415,0.5619
Q7,1.7786,1.7122,0.0351,-1.8919
Q8,1.7605,1.7232,0.0359,-1.0393
Q9,1.7427,1.6872,0.0352,-1.5759
Q10,1.7250,1.6830,0.0365,-1.1527


Average sample latent-score correlation: 0.2470 (calibrated target 0.2332)
Population default-time correlation: 0.200000 (target 0.20)
Average sample default-time correlation: 0.2169
Range across the 45 sample bond-pair correlations: 0.1358 to 0.3070
Sample correlations vary because there are 1,000 scenarios; calibration targets the population.


,Bond 1,Bond 2,Bond 3,Bond 4,Bond 5,Bond 6,Bond 7,Bond 8,Bond 9,Bond 10
Bond 1,1.000,0.155,0.214,0.259,0.255,0.198,0.229,0.176,0.222,0.179
Bond 2,0.155,1.000,0.187,0.225,0.249,0.220,0.160,0.307,0.185,0.237
Bond 3,0.214,0.187,1.000,0.247,0.256,0.212,0.208,0.178,0.194,0.291
Bond 4,0.259,0.225,0.247,1.000,0.286,0.217,0.207,0.209,0.215,0.223
Bond 5,0.255,0.249,0.256,0.286,1.000,0.203,0.252,0.255,0.226,0.262
Bond 6,0.198,0.220,0.212,0.217,0.203,1.000,0.215,0.249,0.136,0.208
Bond 7,0.229,0.160,0.208,0.207,0.252,0.215,1.000,0.193,0.188,0.205
Bond 8,0.176,0.307,0.178,0.209,0.255,0.249,0.193,1.000,0.198,0.216
Bond 9,0.222,0.185,0.194,0.215,0.226,0.136,0.188,0.198,1.000,0.153
Bond 10,0.179,0.237,0.291,0.223,0.262,0.208,0.205,0.216,0.153,1.000


Five-year default frequency: 18.5700%
Theoretical five-year default probability: 18.4627%
Mean number of defaults per scenario: 1.857
Mean total collateral cash ($m): 115.8324
Validation complete: all deterministic cash-flow assertions passed.


### Outputs for the next project section

| Variable | Shape / labels | Contents |
|---|---|---|
| `LATENT_RHO` | Scalar | Calibrated normal-score correlation for population default-time target 0.20 |
| `normal_correlation`, `cholesky_factor` | (10, 10) each | Target normal-score correlation matrix and its Cholesky factor |
| `independent_normals` | (1000, 10) | Seeded independent normal draws, moment-matched columnwise and sorted by case default count |
| `latent_scores`, `sample_latent_correlation` | (1000, 10), (10, 10) | Cholesky-correlated scores and their sample correlation matrix |
| `uniforms` | (1000, 10) | Correlated uniforms obtained with the standard normal CDF |
| `default_times` | (1000, 10) | Exponential default times, sorted worst to best by five-year default count |
| `default_times_df` | Cases 1–1000 × Bonds 1–10 | Labeled ranked default times |
| `bond_cash_flows` | (1000, 10, 20) | Individual bond cash flows in dollars, in ranked case order |
| `coupon_cash_flows`, `recovery_cash_flows`, `principal_cash_flows` | (1000, 10, 20) each | Separate cash flow components, in ranked case order |
| `collateral_cash_flows` | (1000, 20) | Dollar cash available to the waterfall, in ranked case order |
| `collateral_cash_flows_df` | Cases 1–1000 × Q1–Q20 | Labeled collateral cash flows for ranked cases |